# Resume Parser Testing

This notebook tests the new `doc_parser.py` flow in two stages:
1. Parse a resume document into `CandidateProfile` JSON.
2. Run the full resume + cover letter generation pipeline from that document.

## Architecture

The multimodal resume-parser flow in this project is:

```text
Resume File or Image
      |
      v
doc_parser.py
  - detect file type
  - extract text or send image to vision-capable LLM
  - parse into CandidateProfile-shaped JSON
  - normalize and validate against models.py
      |
      v
service.py
  - build prompts from prompts.py
  - call LLM to generate resume markdown
  - call LLM to generate cover letter markdown
      |
      v
Outputs
  - notebooks/outputs/resume_output.md
  - notebooks/outputs/cover_letter_output.md
```

When you test with `.png`, `.jpg`, `.jpeg`, or `.webp`, this is a true multimodal path because the parser sends image content plus instructions to the model. When you test with `.txt`, `.md`, `.docx`, or text-based `.pdf`, it is still the same parser pipeline, but it is primarily text extraction rather than visual parsing.

In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "HireMe.AI-V1").exists():
    repo_root = repo_root.parent

project_dir = repo_root / "HireMe.AI-V1"
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

repo_root, project_dir

(PosixPath('/Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI'),
 PosixPath('/Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/HireMe.AI-V1'))

In [2]:
from dotenv import load_dotenv

load_dotenv(repo_root / ".env", override=True)
print("Loaded .env from:", repo_root / ".env")

Loaded .env from: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/.env


In [3]:
from doc_parser import parse_resume_file
from service import run_pipeline


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
resume_path = project_dir / "sample_resume.txt"
resume_path

PosixPath('/Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/HireMe.AI-V1/sample_resume.txt')

In [5]:
parsed_profile = parse_resume_file(resume_path)
print(json.dumps(parsed_profile, indent=2))

{
  "name": "Taylor Morgan",
  "contact": {
    "email": "taylormorgan@email.com",
    "phone": "(404) 555-0199",
    "website": "www.taylormorgan.dev"
  },
  "summary": "Data analyst and early-career AI product builder with experience translating business needs into dashboards, workflow automations, and candidate-facing tools. Strong interest in job-tech, resume optimization, and LLM-assisted user experiences.",
  "work_experience": [
    {
      "job_title": "Product Analyst",
      "company": "BrightPath Talent",
      "start_date": "June 2024",
      "end_date": "Present",
      "bullets": [
        "Built reporting dashboards that reduced weekly recruiting ops review time by 35%.",
        "Partnered with recruiters and engineers to define metrics for candidate funnel performance.",
        "Tested prompt variations for internal writing assistants used in resume and cover letter generation."
      ]
    },
    {
      "job_title": "Data Analytics Intern",
      "company": "Northst

If the parsed JSON looks correct, run the full pipeline below. Replace the job description text with a real posting when you are ready.

In [6]:
job_description = """
We are hiring a Data Analyst with experience in Python, SQL, dashboarding, stakeholder communication, and process improvement. The role requires translating business needs into reporting, building data-driven insights, and collaborating across recruiting and operations teams.
""".strip()

outputs = run_pipeline(
    repo_root=repo_root,
    candidate_document_path=resume_path,
    job_description=job_description,
    job_title="Data Analyst",
    company_name="Example Company",
    output_dir=repo_root / "notebooks" / "outputs",
)

print(outputs["resume_md"][:1200])
print("\n---\n")
print(outputs["cover_letter_md"][:1200])

# Taylor Morgan
taylormorgan@email.com | (404) 555-0199 | www.taylormorgan.dev

## Summary
Data analyst and early-career AI product builder with experience translating business needs into dashboards, workflow automations, and candidate-facing tools. Proven ability to translate stakeholder questions into data-driven insights, optimize processes, and contribute to AI-enabled recruiting and resume customization initiatives. Strong interest in job-tech, resume optimization, and LLM-assisted user experiences.

## Work Experience
### Product Analyst - BrightPath Talent
June 2024 - Present
- Built reporting dashboards that reduced weekly recruiting ops review time by 35%.
- Partnered with recruiters and engineers to define metrics for candidate funnel performance.
- Tested prompt variations for internal writing assistants used in resume and cover letter generation.

### Data Analytics Intern - Northstar Health
January 2024 - May 2024
- Cleaned and analyzed patient operations data using Python

In [7]:
output_dir = repo_root / "notebooks" / "outputs"
print("Resume output:", output_dir / "resume_output.md")
print("Cover letter output:", output_dir / "cover_letter_output.md")

Resume output: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/resume_output.md
Cover letter output: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/cover_letter_output.md
